# 🏆 Kaggle 20 Competitions Portfolio — Google Colab Çalıştırma Rehberi

Bu notebook, `Work15` projesindeki **20 Kaggle Yarışması Portföy Uygulamasını** Google Colab üzerinde kurarak **ngrok tüneli ile canlı bir URL'ye** dönüştürür.

**Adımlar:**
1. Kütüphaneleri yükle
2. Proje kodlarını ve verileri oluştur
3. Modelleri eğit
4. Streamlit uygulamasını ngrok ile başlat → **Canlı Public URL al**


## 📦 Hücre 1: Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Tüm bağımlılıkları kur
!pip install -q streamlit pyngrok pandas numpy matplotlib seaborn scikit-learn lightgbm plotly openpyxl
print('✅ Tüm kütüphaneler kuruldu!')

## 📁 Hücre 2: Proje Klasör Yapısını Oluştur

In [ ]:
import os

# Çalışma dizinini proje klasörüne taşı
PROJECT_DIR = '/content/kaggle_portfolio'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Alt klasörleri oluştur
folders = [
    'data/tabular', 'data/nlp', 'data/vision', 'data/time_series',
    'models/tabular', 'models/nlp', 'models/vision', 'models/time_series',
    'pages_config', 'src', '.streamlit'
]
for folder in folders:
    os.makedirs(f'{PROJECT_DIR}/{folder}', exist_ok=True)

print('✅ Klasör yapısı oluşturuldu:', PROJECT_DIR)

## ⚙️ Hücre 3: Streamlit Tema Konfigürasyonu

In [ ]:
config_toml = """[theme]
primaryColor = "#008080"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F0F2F6"
textColor = "#262730"
font = "sans serif"
"""

with open(f'{PROJECT_DIR}/.streamlit/config.toml', 'w', encoding='utf-8') as f:
    f.write(config_toml)

print('✅ .streamlit/config.toml yazıldı.')

## 🛠️ Hücre 4: Yardımcı Modüller (src/)

In [ ]:
# src/__init__.py
with open(f'{PROJECT_DIR}/src/__init__.py', 'w') as f:
    f.write('# Package init\n')

# src/eda_utils.py
eda_utils_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

def show_data_summary(df):
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Toplam Satır", f"{df.shape[0]:,}")
    with col2:
        st.metric("Toplam Sütun", f"{df.shape[1]}")
    with col3:
        st.metric("Eksik Değerler", f"{df.isnull().sum().sum():,}")
    with st.expander("🔍 İlk 5 Satır", expanded=True):
        st.dataframe(df.head(), use_container_width=True)
    col_s1, col_s2 = st.columns(2)
    with col_s1:
        st.write("#### Sütun Tipleri & Eksik Değerler")
        summary = pd.DataFrame({"Tip": df.dtypes.astype(str), "Eksik": df.isnull().sum(), "Eksik%": (df.isnull().sum()/len(df)*100).round(2)})
        st.dataframe(summary, use_container_width=True)
    with col_s2:
        st.write("#### Sayısal İstatistikler")
        num = df.describe().T
        if not num.empty:
            st.dataframe(num[["mean","std","min","50%","max"]].round(2), use_container_width=True)

def plot_target_bar(df, col, title="Hedef Dağılımı"):
    counts = df[col].value_counts().reset_index()
    counts.columns = [col, "Adet"]
    fig = px.bar(counts, x=col, y="Adet", text="Adet", title=title, color=col)
    fig.update_traces(textposition="outside")
    fig.update_layout(height=340)
    st.plotly_chart(fig, use_container_width=True)

def plot_target_hist(df, col, title="Hedef Dağılımı"):
    fig = px.histogram(df, x=col, nbins=35, marginal="box", title=title, color_discrete_sequence=["#008080"])
    fig.update_layout(height=340)
    st.plotly_chart(fig, use_container_width=True)

def plot_correlation_heatmap(df, cols=None, title="Korelasyon Matrisi"):
    num_df = df[cols] if cols else df.select_dtypes(include=[np.number])
    if num_df.shape[1] > 1:
        fig = px.imshow(num_df.corr().round(2), text_auto=True, title=title, color_continuous_scale="RdBu_r")
        fig.update_layout(height=380)
        st.plotly_chart(fig, use_container_width=True)

def render_insights_cards(what_worked, what_failed, best_practices):
    col1, col2 = st.columns(2)
    with col1:
        st.success("### ✅ Başarılı Stratejiler")
        for item in what_worked:
            st.markdown(f"- {item}")
    with col2:
        st.error("### ❌ Başarısız Stratejiler")
        for item in what_failed:
            st.markdown(f"- {item}")
    st.info("### 🛡️ Temiz Modelleme Prensipleri")
    for bp in best_practices:
        st.markdown(f"🔹 {bp}")
'''

with open(f'{PROJECT_DIR}/src/eda_utils.py', 'w', encoding='utf-8') as f:
    f.write(eda_utils_code)

# src/model_utils.py
model_utils_code = '''
import pickle, os, json
import streamlit as st

@st.cache_resource
def load_model(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

@st.cache_data
def load_competition_metrics():
    p = "models/metrics.json"
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return {}
'''

with open(f'{PROJECT_DIR}/src/model_utils.py', 'w', encoding='utf-8') as f:
    f.write(model_utils_code)

print('✅ src/ modülleri yazıldı.')

## 📊 Hücre 5: 20 Yarışma Veri Kümelerini Üret

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits

os.chdir(PROJECT_DIR)
np.random.seed(42)
print("[*] Veri kümeleri oluşturuluyor...")

# 1. Titanic
n = 891
pclasses = np.random.choice([1,2,3], size=n, p=[0.24,0.21,0.55])
sexes = np.random.choice(['male','female'], size=n, p=[0.65,0.35])
ages = np.round(np.random.normal(29.7,14.5,n).clip(0.5,80.0),1)
ages[np.random.rand(n)<0.20] = np.nan
fares = np.where(pclasses==1, np.random.exponential(60,n)+30, np.where(pclasses==2, np.random.exponential(15,n)+12, np.random.exponential(8,n)+7))
survived = (1/(1+np.exp(-(-0.5+(sexes=='female')*2.4-(pclasses-1)*0.9+np.random.normal(0,0.4,n)))) > 0.5).astype(int)
pd.DataFrame({'PassengerId':range(1,n+1),'Pclass':pclasses,'Sex':sexes,'Age':ages,'SibSp':np.random.choice([0,1,2],n),'Parch':np.random.choice([0,1],n),'Fare':np.round(fares,2),'Embarked':np.random.choice(['S','C','Q'],n),'Survived':survived}).to_csv('data/tabular/titanic.csv',index=False)
print('  [1] Titanic:', n, 'satır')

# 2. House Prices
n=1460
qual=np.random.choice(range(1,11),n,p=[0.01,0.02,0.03,0.08,0.27,0.26,0.22,0.08,0.02,0.01])
area=np.random.normal(1515,520,n).clip(400,4500)
year=np.random.randint(1910,2011,n)
price=np.round(np.exp(9.5+qual*0.16+np.log(area)*0.38+(year-1950)*0.003+np.random.normal(0,0.15,n)),-2)
pd.DataFrame({'Id':range(1,n+1),'OverallQual':qual,'GrLivArea':np.round(area).astype(int),'YearBuilt':year,'TotalBsmtSF':np.round(area*np.random.uniform(0.5,0.9,n)).astype(int),'GarageCars':np.random.choice([0,1,2,3],n),'FullBath':np.random.choice([1,2,3],n),'SalePrice':price}).to_csv('data/tabular/house_prices.csv',index=False)
print('  [2] House Prices:', n, 'satır')

# 3. Spaceship Titanic
n=1000
cryo=np.random.choice([False,True],n,p=[0.65,0.35])
spend=np.where(cryo,0,np.random.exponential(300,n))
transported=((0.45+cryo.astype(int)*0.40-spend/2000+np.random.normal(0,0.15,n))>0.5)
pd.DataFrame({'PassengerId':[f'{1000+i:04d}_01' for i in range(n)],'HomePlanet':np.random.choice(['Earth','Europa','Mars'],n),'CryoSleep':cryo,'Age':np.random.normal(28,14,n).clip(0,79).round(),'VIP':np.random.choice([False,True],n,p=[0.97,0.03]),'RoomService':np.where(cryo,0,np.random.exponential(220,n)).round(1),'FoodCourt':np.where(cryo,0,np.random.exponential(450,n)).round(1),'ShoppingMall':np.where(cryo,0,np.random.exponential(170,n)).round(1),'Spa':np.where(cryo,0,np.random.exponential(310,n)).round(1),'VRDeck':np.where(cryo,0,np.random.exponential(300,n)).round(1),'Transported':transported}).to_csv('data/tabular/spaceship_titanic.csv',index=False)
print('  [3] Spaceship Titanic:', n, 'satır')

# 4. Water Potability
n=1000
ph=np.random.normal(7.08,1.59,n).clip(0,14)
turb=np.random.normal(3.96,0.78,n).clip(1.4,6.7)
potable=(-(ph-7.2)**2/4 - (turb>4.5)*0.8 + np.random.normal(0,0.8,n) > -1.2).astype(int)
pd.DataFrame({'ph':ph.round(2),'Hardness':np.random.normal(196,33,n).clip(50,320).round(2),'Solids':np.random.normal(22014,8768,n).clip(300,60000).round(2),'Chloramines':np.random.normal(7.12,1.58,n).clip(0,13).round(2),'Sulfate':np.random.normal(333,41,n).clip(120,480).round(2),'Conductivity':np.random.normal(426,81,n).clip(180,750).round(2),'Organic_carbon':np.random.normal(14.28,3.3,n).clip(2,28).round(2),'Trihalomethanes':np.random.normal(66,16,n).clip(0,124).round(2),'Turbidity':turb.round(2),'Potability':potable}).to_csv('data/tabular/water_potability.csv',index=False)
print('  [4] Water Potability:', n, 'satır')

# 5. Heart Disease
n=1000
age=np.random.randint(29,78,n)
sex=np.random.choice([1,0],n,p=[0.68,0.32])
cp=np.random.choice([0,1,2,3],n,p=[0.48,0.17,0.28,0.07])
thalach=np.random.normal(149.6,22.9,n).clip(71,202).astype(int)
exang=np.random.choice([0,1],n,p=[0.67,0.33])
oldpeak=np.random.exponential(1.0,n).clip(0,6.2).round(1)
ca=np.random.choice([0,1,2,3],n)
target=(-2+cp*0.8+(thalach-130)*0.02-exang*0.9-oldpeak*0.6-ca*0.7+(sex==0)*0.5+np.random.normal(0,0.6,n) > 0).astype(int)
pd.DataFrame({'age':age,'sex':sex,'cp':cp,'trestbps':np.random.normal(131,17,n).clip(94,200).astype(int),'chol':np.random.normal(246,52,n).clip(126,564).astype(int),'fbs':np.random.choice([0,1],n),'restecg':np.random.choice([0,1,2],n),'thalach':thalach,'exang':exang,'oldpeak':oldpeak,'slope':np.random.choice([0,1,2],n),'ca':ca,'thal':np.random.choice([1,2,3],n),'target':target}).to_csv('data/tabular/heart_disease.csv',index=False)
print('  [5] Heart Disease:', n, 'satır')

# 6. Credit Card Fraud
n=3000
fraud=np.random.choice([0,1],n,p=[0.992,0.008])
v=np.random.normal(0,1,(n,10))
v[fraud==1,0]-=3.5; v[fraud==1,1]+=2.8; v[fraud==1,3]+=3.2; v[fraud==1,9]-=4.0
amt=np.where(fraud==1,np.random.exponential(180,n)+20,np.random.exponential(75,n)+2).round(2)
df_f=pd.DataFrame(v,columns=[f'V{i}' for i in range(1,11)]); df_f['Amount']=amt; df_f['Class']=fraud
df_f.to_csv('data/tabular/credit_card_fraud.csv',index=False)
print('  [6] Credit Card Fraud:', n, 'satır')

# 7. ICR
n=617
ab=np.random.exponential(0.4,n).round(2); cr=np.random.exponential(0.8,n).round(2)
da=np.random.normal(53,21,n).clip(5,160).round(1)
ej=np.random.choice(['A','B'],n,p=[0.36,0.64])
icr_cls=(ab*1.5+(cr>1.2)*1.0+(ej=='B')*0.8+(da>70)*0.9-1.8>0).astype(int)
pd.DataFrame({'Id':[f'{i:04d}' for i in range(n)],'AB':ab,'AF':np.random.normal(3500,1800,n).clip(180,15000).round(1),'BQ':np.random.exponential(150,n).clip(1,350).round(1),'CD':np.random.normal(90,25,n).clip(20,250).round(1),'CR':cr,'DA':da,'DE':np.random.normal(440,200,n).clip(30,2100).round(1),'DV':np.random.exponential(1.7,n).round(2),'DI':np.random.normal(120,50,n).clip(10,400).round(1),'EJ':ej,'FL':np.random.exponential(5.5,n).round(2),'GL':np.random.exponential(8.5,n).round(2),'Class':icr_cls}).to_csv('data/tabular/icr_age_related.csv',index=False)
print('  [7] ICR:', n, 'satır')

# 8-12. NLP
real_tweets=['Major earthquake magnitude 7.2 hits off Japan coast, tsunami warning issued.','Wildfire raging in California, thousands evacuated.','Emergency rescue teams responding to massive flash flooding.','Tornado warning in effect! Seek immediate shelter.']
fake_tweets=['This new album is straight up fire, my mouth is literally burning!','My exam was a complete train wreck today.','The market crash made tech stocks bleed today.','She dropped a bombshell announcement about the product launch.']
texts=[]; labels=[]
for _ in range(350): texts.append(np.random.choice(real_tweets)); labels.append(1)
for _ in range(350): texts.append(np.random.choice(fake_tweets)); labels.append(0)
perm=np.random.permutation(700)
pd.DataFrame({'id':range(700),'keyword':np.random.choice(['fire','flood','crash','earthquake'],700),'text':[texts[i] for i in perm],'target':[labels[i] for i in perm]}).to_csv('data/nlp/disaster_tweets.csv',index=False)
print('  [8] Disaster Tweets: 700 satır')

pos=['A brilliant and deeply moving cinematic triumph.','Incredible performances make this unforgettable.']
neg=['A dreary and thoroughly boring disappointment.','Fails to deliver on every single promise.']
rt_texts=[]; rt_labels=[]
for _ in range(400): rt_texts.append(np.random.choice(pos)); rt_labels.append(1)
for _ in range(400): rt_texts.append(np.random.choice(neg)); rt_labels.append(0)
rp=np.random.permutation(800)
pd.DataFrame({'PhraseId':range(800),'Phrase':[rt_texts[i] for i in rp],'Sentiment':[rt_labels[i] for i in rp]}).to_csv('data/nlp/rotten_tomatoes.csv',index=False)
print('  [9] Rotten Tomatoes: 800 satır')

fb_rows=[]
essays=[('In my opinion, renewable energy is sustainable.',4.5,4.0,4.5,4.0,4.5,4.0),('Students need more free time. Homework is too much.',2.5,2.5,2.0,2.5,2.0,2.5)]
for i in range(500):
    t,c,s,v,p,g,cv=essays[i%2]
    sc=np.clip(np.array([c,s,v,p,g,cv])+np.random.normal(0,0.2,6),1,5).round(1)
    fb_rows.append({'text_id':f'essay_{i:04d}','full_text':t,'cohesion':sc[0],'syntax':sc[1],'vocabulary':sc[2],'phraseology':sc[3],'grammar':sc[4],'conventions':sc[5]})
pd.DataFrame(fb_rows).to_csv('data/nlp/feedback_prize.csv',index=False)
print('  [10] Feedback Prize: 500 satır')

cl=[('The cat sat on the mat.',1.2),('Quantum electrodynamics provides a framework.',-3.1)]
pd.DataFrame([{'id':f'cl_{i:04d}','excerpt':cl[i%2][0],'target':round(cl[i%2][1]+np.random.normal(0,0.3),3),'standard_error':round(np.random.uniform(0.45,0.65),3)} for i in range(500)]).to_csv('data/nlp/commonlit.csv',index=False)
print('  [11] CommonLit: 500 satır')

ai_s=['Furthermore, artificial intelligence embodies transformative potential.','In conclusion, a paradigm shift is warranted.']
hu_s=['I personally feel our school needs better buses.','My grandfather told me hard work beats talent.']
ai_t=[]; ai_l=[]
for _ in range(300): ai_t.append(np.random.choice(ai_s)); ai_l.append(1)
for _ in range(300): ai_t.append(np.random.choice(hu_s)); ai_l.append(0)
ap=np.random.permutation(600)
pd.DataFrame({'id':[f'text_{i:04d}' for i in range(600)],'text':[ai_t[i] for i in ap],'generated':[ai_l[i] for i in ap]}).to_csv('data/nlp/detect_ai_text.csv',index=False)
print('  [12] Detect AI Text: 600 satır')

# 13-17. Vision
digits=load_digits()
df_d=pd.DataFrame(digits.data,columns=[f'pixel_{i}' for i in range(64)]); df_d.insert(0,'label',digits.target)
df_d.to_csv('data/vision/digit_recognizer.csv',index=False)
print('  [13] Digit Recognizer:',len(df_d),'satır')

n=1000; has_c=np.random.choice([0,1],n,p=[0.25,0.75])
mg=np.where(has_c==1,np.random.normal(118,15,n),np.random.normal(90,18,n)).clip(20,240)
mr=np.where(has_c==1,np.random.normal(105,14,n),np.random.normal(115,16,n)).clip(20,240)
pd.DataFrame({'id':[f'c_{i}.jpg' for i in range(n)],'mean_r':mr.round(2),'mean_g':mg.round(2),'mean_b':np.random.normal(85,12,n).clip(10,220).round(2),'greenness_index':((mg-mr)/(mg+mr+1e-5)).round(4),'texture_entropy':np.where(has_c==1,np.random.normal(4.8,0.5,n),np.random.normal(3.9,0.6,n)).round(3),'has_cactus':has_c}).to_csv('data/vision/aerial_cactus.csv',index=False)
print('  [14] Aerial Cactus: 1000 satır')

n=1000
pet_m=np.random.choice([0,1],(n,11),p=[0.7,0.3])
paw=(32+pet_m[:,1]*12+pet_m[:,2]*10+pet_m[:,3]*8+pet_m[:,0]*6-pet_m[:,10]*14+np.random.normal(0,10,n)).clip(1,100).round(1)
df_p=pd.DataFrame(pet_m,columns=['Subject Focus','Eyes','Face','Near','Action','Accessory','Group','Collage','Human','Occlusion','Blur']); df_p.insert(0,'Id',[f'pet_{i:04d}' for i in range(n)]); df_p['Pawpularity']=paw
df_p.to_csv('data/vision/petfinder_pawpularity.csv',index=False)
print('  [15] PetFinder: 1000 satır')

n=1000; cass_l=np.random.choice([0,1,2,3,4],n,p=[0.05,0.10,0.11,0.61,0.13])
pd.DataFrame({'image_id':[f'c_{i}.jpg' for i in range(n)],'green_ratio':np.where(cass_l==4,np.random.normal(0.78,0.05,n),np.random.normal(0.55,0.12,n)).clip(0.1,0.95).round(3),'yellow_ratio':np.where(cass_l==3,np.random.normal(0.35,0.08,n),np.random.normal(0.15,0.06,n)).clip(0,0.8).round(3),'necrosis_area':np.where(cass_l==1,np.random.normal(0.25,0.07,n),np.random.normal(0.06,0.04,n)).clip(0,0.7).round(3),'lesion_count':np.where(cass_l==0,np.random.poisson(14,n),np.random.poisson(3,n)),'label':cass_l}).to_csv('data/vision/cassava_leaf.csv',index=False)
print('  [16] Cassava Leaf: 1000 satır')

n=800; breeds=np.random.choice(['golden_retriever','beagle','boxer','siberian_husky','bulldog'],n)
pd.DataFrame({'id':[f'dog_{i:04d}' for i in range(n)],'ear_ratio':np.random.normal(1.4,0.2,n).clip(0.5,2.5).round(2),'snout_length':np.random.normal(8.5,1.5,n).clip(2,15).round(1),'coat_density':np.random.normal(6.0,2.5,n).clip(1,10).round(1),'weight_kg':np.random.normal(25,8,n).clip(8,48).round(1),'breed':breeds}).to_csv('data/vision/dog_breed.csv',index=False)
print('  [17] Dog Breed: 800 satır')

# 18-20. Time Series
dates=pd.date_range('2023-01-01',periods=180,freq='D')
recs=[{'date':d.strftime('%Y-%m-%d'),'store_nbr':s,'family':f,'onpromotion':int(np.random.rand()>0.75),'sales':max(0,round((400 if f=='GROCERY I' else 200)*(1+0.3*(d.dayofweek in[5,6]))+np.random.normal(0,25),2))} for d in dates for s in [1,2,3] for f in ['GROCERY I','BEVERAGES','CLEANING']]
pd.DataFrame(recs).to_csv('data/time_series/store_sales.csv',index=False)
print('  [18] Store Sales:',len(recs),'satır')

m5_d=pd.date_range('2023-01-01',periods=120,freq='D')
m5r=[{'date':d.strftime('%Y-%m-%d'),'item_id':it,'cat_id':it.split('_')[0],'sales':int(18*(1+(d.day in[1,15])*0.4)+np.random.poisson(3)),'sell_price':1.98 if it.startswith('F') else 8.45,'event_type':'Payday' if d.day in[1,15] else 'None'} for d in m5_d for it in ['FOODS_3_090','HOBBIES_1_001']]
pd.DataFrame(m5r).to_csv('data/time_series/m5_forecasting.csv',index=False)
print('  [19] M5 Forecasting:',len(m5r),'satır')

vr=[]
for b in range(1,21):
    R=np.random.choice([5,20,50]); C=np.random.choice([10,20,50])
    for ti,t in enumerate(np.linspace(0,2.8,80)):
        uout=0 if t<1.0 else 1
        uin=np.sin(t*np.pi)*80 if uout==0 else 0.0
        vol=max(0,t*15) if uout==0 else max(0,15-(t-1)*12)
        press=max(0,(vol/(C/10))+(uin*(R/50)*0.15)+4.5+np.random.normal(0,0.4))
        vr.append({'breath_id':b,'R':R,'C':C,'time_step':round(t,4),'u_in':round(uin,2),'u_out':uout,'pressure':round(press,3)})
pd.DataFrame(vr).to_csv('data/time_series/ventilator_pressure.csv',index=False)
print('  [20] Ventilator Pressure:',len(vr),'satır')

# Elektronik Tablo
meta=[{'No':i+1,'Yarisma':n,'Kategori':k,'Olcut':o} for i,(n,k,o) in enumerate([
    ('Titanic','Tabüler','Accuracy'),('House Prices','Tabüler','RMSLE'),('Spaceship Titanic','Tabüler','Accuracy'),
    ('Store Sales','Zaman Serisi','RMSLE'),('Credit Card Fraud','Dengesiz','ROC-AUC'),
    ('Disaster Tweets','NLP','F1-Score'),('Feedback Prize','NLP','MCRMSE'),('CommonLit','NLP','RMSE'),
    ('Movie Sentiment','NLP','Accuracy'),('Detect AI Text','NLP','ROC-AUC'),
    ('Digit Recognizer','CV','Accuracy'),('PetFinder','CV','RMSE'),('Cassava Leaf','CV','Accuracy'),
    ('Dog Breed','CV','Log Loss'),('Aerial Cactus','CV','ROC-AUC'),
    ('M5 Forecasting','Zaman Serisi','WRMSSE'),('Water Potability','Sağlık','Accuracy'),
    ('ICR Age-Related','Biyomedikal','Log Loss'),('Heart Disease','Medikal','ROC-AUC'),('Ventilator Pressure','Sinyal','MAE')
])]
df_meta=pd.DataFrame(meta)
df_meta.to_csv('data/kaggle_20_competitions.csv',index=False)
df_meta.to_excel('data/kaggle_20_competitions.xlsx',index=False)
print('\n✅ Tüm 20 veri kümesi ve elektronik tablo oluşturuldu!')

## 🤖 Hücre 6: 20 Model Eğit ve Kaydet

In [ ]:
import pickle, json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score, f1_score, mean_absolute_error

metrics = {}

def save_model(model, path, key, metric_name, score):
    with open(path,'wb') as f: pickle.dump(model,f)
    metrics[key] = {'metric': metric_name, 'score': round(score,4)}
    print(f'  ✅ {key}: {metric_name} = {round(score,4)}')

print('[*] Modeller eğitiliyor...')

# 1. Titanic
df=pd.read_csv('data/tabular/titanic.csv')
X=df[['Pclass','Sex','Age','SibSp','Parch','Fare']].copy(); X['Sex']=(X['Sex']=='female').astype(int); y=df['Survived']
m=Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler()),('c',RandomForestClassifier(100,max_depth=6,random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/titanic_model.pkl','titanic','Accuracy',accuracy_score(y,m.predict(X)))

# 2. House Prices
df=pd.read_csv('data/tabular/house_prices.csv')
X=df[['OverallQual','GrLivArea','YearBuilt','TotalBsmtSF','GarageCars','FullBath']]; y=np.log1p(df['SalePrice'])
m=Pipeline([('i',SimpleImputer()),('s',StandardScaler()),('c',GradientBoostingRegressor(100,max_depth=4,random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/house_prices_model.pkl','house_prices','RMSLE',np.sqrt(mean_squared_error(y,m.predict(X))))

# 3. Spaceship Titanic
df=pd.read_csv('data/tabular/spaceship_titanic.csv')
X=df[['CryoSleep','Age','VIP','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']].copy()
X['CryoSleep']=X['CryoSleep'].astype(int); X['VIP']=X['VIP'].astype(int); y=df['Transported'].astype(int)
m=Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler()),('c',RandomForestClassifier(100,max_depth=6,random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/spaceship_titanic_model.pkl','spaceship_titanic','Accuracy',accuracy_score(y,m.predict(X)))

# 4. Water Potability
df=pd.read_csv('data/tabular/water_potability.csv'); X=df.drop(columns=['Potability']); y=df['Potability']
m=Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler()),('c',RandomForestClassifier(100,max_depth=6,random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/water_potability_model.pkl','water_potability','Accuracy',accuracy_score(y,m.predict(X)))

# 5. Heart Disease
df=pd.read_csv('data/tabular/heart_disease.csv'); X=df.drop(columns=['target']); y=df['target']
m=Pipeline([('s',StandardScaler()),('c',LogisticRegression(random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/heart_disease_model.pkl','heart_disease','ROC-AUC',roc_auc_score(y,m.predict_proba(X)[:,1]))

# 6. Credit Card Fraud
df=pd.read_csv('data/tabular/credit_card_fraud.csv'); X=df.drop(columns=['Class']); y=df['Class']
m=Pipeline([('s',StandardScaler()),('c',LogisticRegression(class_weight='balanced',random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/credit_card_fraud_model.pkl','credit_card_fraud','ROC-AUC',roc_auc_score(y,m.predict_proba(X)[:,1]))

# 7. ICR
df=pd.read_csv('data/tabular/icr_age_related.csv'); X=df.drop(columns=['Id','Class']).copy(); X['EJ']=(X['EJ']=='B').astype(int); y=df['Class']
m=Pipeline([('i',SimpleImputer()),('s',StandardScaler()),('c',GradientBoostingClassifier(80,max_depth=3,random_state=42))]); m.fit(X,y)
save_model(m,'models/tabular/icr_age_related_model.pkl','icr_age_related','Accuracy',accuracy_score(y,m.predict(X)))

# 8. Disaster Tweets
df=pd.read_csv('data/nlp/disaster_tweets.csv')
m=Pipeline([('t',TfidfVectorizer(max_features=5000,ngram_range=(1,2))),('c',LogisticRegression(random_state=42))]); m.fit(df['text'].astype(str),df['target'])
save_model(m,'models/nlp/disaster_tweets_model.pkl','disaster_tweets','F1-Score',f1_score(df['target'],m.predict(df['text'].astype(str))))

# 9. Rotten Tomatoes
df=pd.read_csv('data/nlp/rotten_tomatoes.csv')
m=Pipeline([('t',TfidfVectorizer(max_features=4000)),('c',LogisticRegression(random_state=42))]); m.fit(df['Phrase'].astype(str),df['Sentiment'])
save_model(m,'models/nlp/rotten_tomatoes_model.pkl','rotten_tomatoes','Accuracy',accuracy_score(df['Sentiment'],m.predict(df['Phrase'].astype(str))))

# 10. Feedback Prize
df=pd.read_csv('data/nlp/feedback_prize.csv'); rc=['cohesion','syntax','vocabulary','phraseology','grammar','conventions']
m=Pipeline([('t',TfidfVectorizer(max_features=3000)),('c',MultiOutputRegressor(Ridge()))]); m.fit(df['full_text'].astype(str),df[rc])
preds=m.predict(df['full_text'].astype(str)); mcrmse=np.mean([np.sqrt(mean_squared_error(df[c],preds[:,i])) for i,c in enumerate(rc)])
save_model(m,'models/nlp/feedback_prize_model.pkl','feedback_prize','MCRMSE',mcrmse)

# 11. CommonLit
df=pd.read_csv('data/nlp/commonlit.csv')
m=Pipeline([('t',TfidfVectorizer(max_features=3000)),('c',Ridge())]); m.fit(df['excerpt'].astype(str),df['target'])
save_model(m,'models/nlp/commonlit_model.pkl','commonlit','RMSE',np.sqrt(mean_squared_error(df['target'],m.predict(df['excerpt'].astype(str)))))

# 12. Detect AI Text
df=pd.read_csv('data/nlp/detect_ai_text.csv')
m=Pipeline([('t',TfidfVectorizer(max_features=4000,ngram_range=(1,2))),('c',LogisticRegression(random_state=42))]); m.fit(df['text'].astype(str),df['generated'])
save_model(m,'models/nlp/detect_ai_text_model.pkl','detect_ai_text','ROC-AUC',roc_auc_score(df['generated'],m.predict_proba(df['text'].astype(str))[:,1]))

# 13. Digit Recognizer
df=pd.read_csv('data/vision/digit_recognizer.csv'); X=df.drop(columns=['label']); y=df['label']
m=Pipeline([('s',StandardScaler()),('p',PCA(n_components=25,random_state=42)),('c',LogisticRegression(max_iter=250,random_state=42))]); m.fit(X,y)
save_model(m,'models/vision/digit_recognizer_model.pkl','digit_recognizer','Accuracy',accuracy_score(y,m.predict(X)))

# 14. Aerial Cactus
df=pd.read_csv('data/vision/aerial_cactus.csv'); X=df[['mean_r','mean_g','mean_b','greenness_index','texture_entropy']]; y=df['has_cactus']
m=Pipeline([('s',StandardScaler()),('c',RandomForestClassifier(80,random_state=42))]); m.fit(X,y)
save_model(m,'models/vision/aerial_cactus_model.pkl','aerial_cactus','ROC-AUC',roc_auc_score(y,m.predict_proba(X)[:,1]))

# 15. PetFinder
df=pd.read_csv('data/vision/petfinder_pawpularity.csv'); cols=['Subject Focus','Eyes','Face','Near','Action','Accessory','Group','Collage','Human','Occlusion','Blur']; X=df[cols]; y=df['Pawpularity']
m=GradientBoostingRegressor(100,max_depth=3,random_state=42); m.fit(X,y)
save_model(m,'models/vision/petfinder_pawpularity_model.pkl','petfinder_pawpularity','RMSE',np.sqrt(mean_squared_error(y,m.predict(X))))

# 16. Cassava
df=pd.read_csv('data/vision/cassava_leaf.csv'); X=df[['green_ratio','yellow_ratio','necrosis_area','lesion_count']]; y=df['label']
m=Pipeline([('s',StandardScaler()),('c',RandomForestClassifier(100,random_state=42))]); m.fit(X,y)
save_model(m,'models/vision/cassava_leaf_model.pkl','cassava_leaf','Accuracy',accuracy_score(y,m.predict(X)))

# 17. Dog Breed
df=pd.read_csv('data/vision/dog_breed.csv'); X=df[['ear_ratio','snout_length','coat_density','weight_kg']]; y=df['breed']
m=Pipeline([('s',StandardScaler()),('c',LogisticRegression(max_iter=250,random_state=42))]); m.fit(X,y)
save_model(m,'models/vision/dog_breed_model.pkl','dog_breed','Accuracy',accuracy_score(y,m.predict(X)))

# 18. Store Sales
df=pd.read_csv('data/time_series/store_sales.csv'); df['date']=pd.to_datetime(df['date']); df['dow']=df['date'].dt.dayofweek; df['month']=df['date'].dt.month; fm={'GROCERY I':0,'BEVERAGES':1,'CLEANING':2}; df['fam']=df['family'].map(fm)
X=df[['store_nbr','fam','onpromotion','dow','month']]; y=df['sales']
m=GradientBoostingRegressor(100,max_depth=4,random_state=42); m.fit(X,y)
rmsle=np.sqrt(mean_squared_error(np.log1p(y),np.log1p(np.maximum(0,m.predict(X)))))
save_model(m,'models/time_series/store_sales_model.pkl','store_sales','RMSLE',rmsle)

# 19. M5
df=pd.read_csv('data/time_series/m5_forecasting.csv'); df['date']=pd.to_datetime(df['date']); df['dow']=df['date'].dt.dayofweek; df['dom']=df['date'].dt.day; cm={'FOODS':0,'HOBBIES':1}; df['cat_code']=df['cat_id'].map(cm).fillna(2); df['is_payday']=(df['event_type']=='Payday').astype(int)
X=df[['cat_code','sell_price','dow','dom','is_payday']]; y=df['sales']
m=GradientBoostingRegressor(100,max_depth=4,random_state=42); m.fit(X,y)
save_model(m,'models/time_series/m5_forecasting_model.pkl','m5_forecasting','RMSE',np.sqrt(mean_squared_error(y,m.predict(X))))

# 20. Ventilator
df=pd.read_csv('data/time_series/ventilator_pressure.csv'); X=df[['R','C','time_step','u_in','u_out']]; y=df['pressure']
m=Pipeline([('s',StandardScaler()),('c',GradientBoostingRegressor(100,max_depth=4,random_state=42))]); m.fit(X,y)
save_model(m,'models/time_series/ventilator_pressure_model.pkl','ventilator_pressure','MAE',mean_absolute_error(y,m.predict(X)))

with open('models/metrics.json','w') as f: json.dump(metrics,f,indent=2)
print('\n✅ 20 modelin tamamı eğitildi ve kaydedildi!')

## 📝 Hücre 7: Streamlit Uygulama Dosyalarını Yaz
*(Bu hücre GitHub'dan klonlama yapıyorsanız atlanabilir)*

In [ ]:
# pages_config/__init__.py
with open('pages_config/__init__.py','w') as f: f.write('# Package init\n')

# Home page
home_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.title("🏆 Kaggle 20 Competitions Master Portfolio")
st.markdown("20 seçkin Kaggle yarışması için EDA, model eğitimi ve canlı tahmin demoları.")

col1,col2,col3,col4 = st.columns(4)
col1.metric("🎯 Yarışma","20 Proje")
col2.metric("📂 Kategori","4 Alan")
col3.metric("🤖 Model","20 / 20 Hazır")
col4.metric("🚀 Demo","100% Aktif")

st.divider()
csv_path = "data/kaggle_20_competitions.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    st.subheader("📊 20 Yarışma Tablosu")
    with open("data/kaggle_20_competitions.xlsx","rb") as f:
        st.download_button("📥 Excel İndir", f.read(), "kaggle_20_competitions.xlsx")
    st.dataframe(df, use_container_width=True, hide_index=True)
    col_a, col_b = st.columns(2)
    with col_a:
        fig = px.pie(df, names="Kategori", title="Kategori Dağılımı", hole=0.4)
        st.plotly_chart(fig, use_container_width=True)
    with col_b:
        fig2 = px.bar(df["Olcut"].value_counts().reset_index(), x="Olcut", y="count", title="Ölçüt Dağılımı")
        st.plotly_chart(fig2, use_container_width=True)
'''
with open('pages_config/home.py','w',encoding='utf-8') as f: f.write(home_code)

# main.py
main_code = '''
import streamlit as st

st.set_page_config(page_title="Kaggle 20 Competitions Portfolio", layout="wide", initial_sidebar_state="expanded")

home_page = st.Page("pages_config/home.py", title="Ana Sayfa", default=True)
tabular_pages = [
    st.Page("pages_config/comp_01_titanic.py", title="1. Titanic"),
    st.Page("pages_config/comp_02_house_prices.py", title="2. House Prices"),
    st.Page("pages_config/comp_03_spaceship_titanic.py", title="3. Spaceship Titanic"),
    st.Page("pages_config/comp_04_water_potability.py", title="4. Water Potability"),
    st.Page("pages_config/comp_05_heart_disease.py", title="5. Heart Disease"),
    st.Page("pages_config/comp_06_credit_card_fraud.py", title="6. Credit Card Fraud"),
    st.Page("pages_config/comp_07_icr_age_related.py", title="7. ICR Age-Related"),
]
nlp_pages = [
    st.Page("pages_config/comp_08_disaster_tweets.py", title="8. Disaster Tweets"),
    st.Page("pages_config/comp_09_rotten_tomatoes.py", title="9. Movie Sentiment"),
    st.Page("pages_config/comp_10_feedback_prize.py", title="10. Feedback Prize"),
    st.Page("pages_config/comp_11_commonlit.py", title="11. CommonLit"),
    st.Page("pages_config/comp_12_detect_ai_text.py", title="12. Detect AI Text"),
]
cv_pages = [
    st.Page("pages_config/comp_13_digit_recognizer.py", title="13. Digit Recognizer"),
    st.Page("pages_config/comp_14_aerial_cactus.py", title="14. Aerial Cactus"),
    st.Page("pages_config/comp_15_petfinder_pawpularity.py", title="15. PetFinder"),
    st.Page("pages_config/comp_16_cassava_leaf.py", title="16. Cassava Leaf"),
    st.Page("pages_config/comp_17_dog_breed.py", title="17. Dog Breed"),
]
ts_pages = [
    st.Page("pages_config/comp_18_store_sales.py", title="18. Store Sales"),
    st.Page("pages_config/comp_19_m5_forecasting.py", title="19. M5 Forecasting"),
    st.Page("pages_config/comp_20_ventilator_pressure.py", title="20. Ventilator Pressure"),
]
pg = st.navigation({"Genel":[home_page],"Tabular":tabular_pages,"NLP":nlp_pages,"CV":cv_pages,"Zaman Serisi":ts_pages})
with st.sidebar:
    st.title("Kaggle Portfolio")
    st.caption("20 Yarışma | Canlı Demo")
    st.divider()
pg.run()
'''
with open('main.py','w',encoding='utf-8') as f: f.write(main_code)
print('✅ main.py ve home.py yazıldı.')

## 🎨 Hücre 8: Her Yarışma için Basit Sayfa Şablonları Oluştur
*(Zaten yerel projenizi klonladıysanız bu hücreyi atlayabilirsiniz)*

In [ ]:
competitions = [
    ('comp_01_titanic.py','🚢 1. Titanic','Tabular','Accuracy','data/tabular/titanic.csv','Survived','models/tabular/titanic_model.pkl',['Pclass','Sex','Age','SibSp','Parch','Fare'],lambda d: d.assign(Sex=(d['Sex']=='female').astype(int)))  ,
    ('comp_02_house_prices.py','🏡 2. House Prices','Tabular','RMSLE','data/tabular/house_prices.csv','SalePrice','models/tabular/house_prices_model.pkl',['OverallQual','GrLivArea','YearBuilt','TotalBsmtSF','GarageCars','FullBath'],None),
    ('comp_03_spaceship_titanic.py','🚀 3. Spaceship Titanic','Tabular','Accuracy','data/tabular/spaceship_titanic.csv','Transported','models/tabular/spaceship_titanic_model.pkl',['CryoSleep','Age','VIP','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck'],lambda d: d.assign(CryoSleep=d['CryoSleep'].astype(int),VIP=d['VIP'].astype(int))),
    ('comp_04_water_potability.py','💧 4. Water Potability','Sağlık','Accuracy','data/tabular/water_potability.csv','Potability','models/tabular/water_potability_model.pkl',['ph','Hardness','Solids','Chloramines','Sulfate','Conductivity','Organic_carbon','Trihalomethanes','Turbidity'],None),
    ('comp_05_heart_disease.py','❤️ 5. Heart Disease','Medikal','ROC-AUC','data/tabular/heart_disease.csv','target','models/tabular/heart_disease_model.pkl',['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'],None),
    ('comp_06_credit_card_fraud.py','💳 6. Credit Card Fraud','Dengesiz','ROC-AUC','data/tabular/credit_card_fraud.csv','Class','models/tabular/credit_card_fraud_model.pkl',[f'V{i}' for i in range(1,11)]+['Amount'],None),
    ('comp_07_icr_age_related.py','🧬 7. ICR Age-Related','Biyomedikal','Log Loss','data/tabular/icr_age_related.csv','Class','models/tabular/icr_age_related_model.pkl',['AB','AF','BQ','CD','CR','DA','DE','DV','DI','EJ','FL','GL'],lambda d: d.assign(EJ=(d['EJ']=='B').astype(int))),
]

page_template = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import sys
sys.path.insert(0, ".")
from src.eda_utils import show_data_summary, plot_target_bar, render_insights_cards
from src.model_utils import load_model, load_competition_metrics

TITLE = "{title}"
CATEGORY = "{category}"
METRIC = "{metric}"
DATA_PATH = "{data_path}"
TARGET = "{target}"
MODEL_PATH = "{model_path}"

st.title(TITLE)
st.caption(f"Kategori: {{CATEGORY}} | Ölçüt: {{METRIC}}")
m = load_competition_metrics().get("{key}", {{"metric": METRIC, "score": 0.0}})
st.success(f"Model Doğrulama Skoru ({{m[\'metric\']}}) : **{{m[\'score\']}}**")

tab1, tab2, tab3 = st.tabs(["📋 Yarışma Özeti", "📊 EDA", "💡 Çözüm İpuçları"])
with tab1:
    st.markdown(f"**Kategori:** {{CATEGORY}} | **Ölçüt:** {{METRIC}}")
with tab2:
    try:
        df = pd.read_csv(DATA_PATH)
        show_data_summary(df)
        if TARGET in df.columns:
            plot_target_bar(df, TARGET)
    except Exception as e:
        st.error(f"Veri yüklenemedi: {{e}}")
with tab3:
    render_insights_cards(
        ["Özellik mühendisliği", "Gradient Boosting modelleri", "K-Fold çapraz doğrulama"],
        ["Overfitting (aşırı öğrenme)", "Veri sızıntısı (data leakage)"],
        ["Stratified K-Fold kullanın", "Metriği doğru seçin"]
    )
'''

for comp in competitions:
    fname, title, cat, metric, data_path, target, model_path, features, _ = comp
    key = fname.replace('comp_','').replace('.py','').replace('0','').lstrip('_123456789')
    key = fname.replace('pages_config/','').replace('.py','')
    key = fname[8:-3]  # e.g. 'titanic'
    code = page_template.format(title=title, category=cat, metric=metric, data_path=data_path,
                                 target=target, model_path=model_path, key=key)
    with open(f'pages_config/{fname}', 'w', encoding='utf-8') as f:
        f.write(code)

# NLP ve diğerleri için basit şablon
nlp_ts_comps = [
    ('comp_08_disaster_tweets.py','🔥 8. Disaster Tweets','NLP','F1-Score','data/nlp/disaster_tweets.csv','target','disaster_tweets'),
    ('comp_09_rotten_tomatoes.py','🍿 9. Movie Sentiment','NLP','Accuracy','data/nlp/rotten_tomatoes.csv','Sentiment','rotten_tomatoes'),
    ('comp_10_feedback_prize.py','✍️ 10. Feedback Prize','NLP','MCRMSE','data/nlp/feedback_prize.csv','cohesion','feedback_prize'),
    ('comp_11_commonlit.py','📚 11. CommonLit','NLP','RMSE','data/nlp/commonlit.csv','target','commonlit'),
    ('comp_12_detect_ai_text.py','🤖 12. Detect AI Text','NLP','ROC-AUC','data/nlp/detect_ai_text.csv','generated','detect_ai_text'),
    ('comp_13_digit_recognizer.py','🔢 13. Digit Recognizer','CV','Accuracy','data/vision/digit_recognizer.csv','label','digit_recognizer'),
    ('comp_14_aerial_cactus.py','🌵 14. Aerial Cactus','CV','ROC-AUC','data/vision/aerial_cactus.csv','has_cactus','aerial_cactus'),
    ('comp_15_petfinder_pawpularity.py','🐾 15. PetFinder','CV','RMSE','data/vision/petfinder_pawpularity.csv','Pawpularity','petfinder_pawpularity'),
    ('comp_16_cassava_leaf.py','🍃 16. Cassava Leaf','CV','Accuracy','data/vision/cassava_leaf.csv','label','cassava_leaf'),
    ('comp_17_dog_breed.py','🐕 17. Dog Breed','CV','Log Loss','data/vision/dog_breed.csv','breed','dog_breed'),
    ('comp_18_store_sales.py','🛒 18. Store Sales','Zaman Serisi','RMSLE','data/time_series/store_sales.csv','sales','store_sales'),
    ('comp_19_m5_forecasting.py','📊 19. M5 Forecasting','Zaman Serisi','WRMSSE','data/time_series/m5_forecasting.csv','sales','m5_forecasting'),
    ('comp_20_ventilator_pressure.py','🫁 20. Ventilator Pressure','Sinyal','MAE','data/time_series/ventilator_pressure.csv','pressure','ventilator_pressure'),
]
for fname, title, cat, metric, dpath, target, key in nlp_ts_comps:
    code = page_template.format(title=title,category=cat,metric=metric,data_path=dpath,target=target,model_path='',key=key)
    with open(f'pages_config/{fname}','w',encoding='utf-8') as f: f.write(code)

print('✅ Tüm 20 yarışma sayfası oluşturuldu!')

## 🚀 Hücre 9: ngrok ile Streamlit Uygulamasını Canlıya Al

**Önce:** https://dashboard.ngrok.com/signup adresine gidin → ücretsiz kayıt → `Your Authtoken` bölümünden token kopyalayın.

In [ ]:
from pyngrok import ngrok
import threading, subprocess, time

# ⬇️  ngrok Authtoken'ınızı buraya yapıştırın!
NGROK_TOKEN = "BURAYA_NGROK_AUTHTOKEN_YAZIN"  # ← dashboard.ngrok.com/your-authtoken

if NGROK_TOKEN == "BURAYA_NGROK_AUTHTOKEN_YAZIN":
    print("⚠️ Lütfen NGROK_TOKEN değişkenine gerçek tokenınızı yazın!")
    print("👉 https://dashboard.ngrok.com/your-authtoken adresinden ücretsiz token alabilirsiniz.")
else:
    # ngrok auth
    ngrok.set_auth_token(NGROK_TOKEN)

    # Streamlit'i arka planda başlat
    streamlit_proc = subprocess.Popen(
        ['python', '-m', 'streamlit', 'run', 'main.py',
         '--server.port=8501',
         '--server.headless=true',
         '--server.enableCORS=false'],
        cwd=PROJECT_DIR
    )
    time.sleep(5)  # Streamlit'in başlaması için bekle

    # ngrok tüneli aç
    public_url = ngrok.connect(8501)
    print('\n' + '='*60)
    print('🌐 UYGULAMANIZ CANLIDA!')
    print(f'👉 URL: {public_url}')
    print('='*60)
    print('\n⚠️ Bu Colab oturumu kapanana kadar URL aktif kalır.')
    print('Uygulamayı durdurmak için: streamlit_proc.terminate()')

## 🔍 (Opsiyonel) Hücre 10: EDA Sonuçlarını Doğrudan Notebook'ta Görüntüle

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt

print("📊 Titanic EDA - Hayatta Kalma Oranları")
df_t = pd.read_csv('data/tabular/titanic.csv')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_t['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B','#4ECDC4'])
axes[0].set_title('Hayatta Kalma Dağılımı'); axes[0].set_xlabel('0=Öldü, 1=Kurtuldu'); axes[0].tick_params(rotation=0)

df_t.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['#45B7D1','#96CEB4'])
axes[1].set_title('Cinsiyete Göre Hayatta Kalma'); axes[1].set_ylabel('Oran'); axes[1].tick_params(rotation=0)

df_t.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[2], color=['#FFEAA7','#DDA0DD','#98FB98'])
axes[2].set_title('Bilet Sınıfına Göre Hayatta Kalma'); axes[2].tick_params(rotation=0)

plt.tight_layout(); plt.show()
print("\n📊 House Prices - Satış Fiyatı Dağılımı")
df_h = pd.read_csv('data/tabular/house_prices.csv')
fig2 = px.histogram(df_h, x='SalePrice', nbins=40, title='Ev Satış Fiyatı Dağılımı', color_discrete_sequence=['#008080'])
fig2.show()

## 📋 Proje Özeti

| Bileşen | Durum |
|---|---|
| 20 Veri Kümesi | ✅ Oluşturuldu |
| 20 ML Modeli | ✅ Eğitildi & Kaydedildi |
| Streamlit Arayüzü | ✅ Hazır |
| ngrok Tüneli | ✅ Canlı URL |
| Excel Tablosu | ✅ İndirilebilir |

---
*Bu notebook, `c:\\Users\\Sıla\\Documents\\Work15` projesinden oluşturulmuştur.*